# 04 - Reconcile the Admin Monitoring archive

Validates an extraction run against the manifest and persisted Delta tables. The notebook fails if a table was skipped, extraction failed, source/extracted counts differ, schema evidence is missing, or a target Delta table is absent.


In [ ]:
run_id = ""
target_workspace_name = "YOUR_TARGET_WORKSPACE"
lakehouse_name = "AdminMonitoringArchive"
data_schema = "admin_monitoring"
control_schema = "admin_monitoring_control"


In [ ]:
import json
from datetime import datetime, timezone

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F


def fabric_get(path):
    response = requests.get(
        f"https://api.fabric.microsoft.com/v1/{path.lstrip('/')}",
        headers={
            "Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}",
            "x-ms-fabric-skill": "spark-cli",
        },
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"Fabric GET {path} failed: {response.status_code} {response.text[:2000]}")
    return response.json()


context = notebookutils.runtime.context
workspace_id = str(context.get("currentWorkspaceId") or "")
workspace_name = str(context.get("currentWorkspaceName") or "")
if not workspace_id or workspace_name.casefold() != target_workspace_name.casefold():
    raise RuntimeError(
        f'This notebook must run in "{target_workspace_name}". Current workspace: "{workspace_name}".'
    )

lakehouses = fabric_get(f"workspaces/{workspace_id}/lakehouses").get("value", [])
matches = [item for item in lakehouses if item.get("displayName", "").casefold() == lakehouse_name.casefold()]
if len(matches) != 1:
    raise RuntimeError(f'Expected one Lakehouse named "{lakehouse_name}"; found {len(matches)}.')
lakehouse_id = str(matches[0]["id"])
endpoint = notebookutils.conf.get("trident.onelake.endpoint").removeprefix("https://").removeprefix("http://").rstrip("/")
table_root = f"abfss://{workspace_id}@{endpoint}/{lakehouse_id}/Tables"
control_root = f"{table_root}/{control_schema}"
data_root = f"{table_root}/{data_schema}"

manifest_path = f"{control_root}/table_manifest"
audit_path = f"{control_root}/run_audit"
schema_path = f"{control_root}/schema_history"
reconciliation_path = f"{control_root}/reconciliation"
for required_path in (manifest_path, audit_path, schema_path, reconciliation_path):
    if not DeltaTable.isDeltaTable(spark, required_path):
        raise RuntimeError(f"Required control table is missing: {required_path}")

audit = spark.read.format("delta").load(audit_path)
selected_run_id = run_id.strip()
if not selected_run_id:
    latest = audit.orderBy(F.col("ended_utc").desc()).select("run_id").limit(1).collect()
    if not latest:
        raise RuntimeError("No extraction run exists to reconcile.")
    selected_run_id = latest[0]["run_id"]

run_rows = audit.filter(F.col("run_id") == selected_run_id).cache()
manifest = spark.read.format("delta").load(manifest_path)
schema_history = spark.read.format("delta").load(schema_path)
expected_count = manifest.count()
actual_count = run_rows.count()
failed_count = run_rows.filter(F.col("status") != "Succeeded").count()
mismatch_count = run_rows.filter(
    F.col("source_row_count").isNull()
    | F.col("extracted_row_count").isNull()
    | (F.col("source_row_count") != F.col("extracted_row_count"))
).count()

expected_keys = {
    (row["semantic_model"], row["source_table"])
    for row in manifest.select("semantic_model", "source_table").collect()
}
actual_keys = {
    (row["semantic_model"], row["source_table"])
    for row in run_rows.select("semantic_model", "source_table").collect()
}
missing_keys = sorted(expected_keys - actual_keys)

missing_targets = []
for row in run_rows.select("target_table").distinct().collect():
    target_path = f"{data_root}/{row['target_table']}"
    if not DeltaTable.isDeltaTable(spark, target_path):
        missing_targets.append(row["target_table"])

schema_count = schema_history.filter(F.col("run_id") == selected_run_id).count()
schema_changes = (
    schema_history
    .groupBy("semantic_model", "source_table")
    .agg(F.countDistinct("schema_hash").alias("schema_versions"))
    .filter(F.col("schema_versions") > 1)
    .count()
)

checks = [
    ("manifest_row_count", actual_count == expected_count, f"expected={expected_count}, actual={actual_count}"),
    ("manifest_key_coverage", not missing_keys, json.dumps(missing_keys)),
    ("table_status", failed_count == 0, f"failed_tables={failed_count}"),
    ("source_extract_row_parity", mismatch_count == 0, f"mismatched_tables={mismatch_count}"),
    ("target_delta_tables", not missing_targets, json.dumps(missing_targets)),
    ("schema_evidence", schema_count == expected_count, f"expected={expected_count}, actual={schema_count}"),
    ("schema_drift_observation", True, f"tables_with_multiple_observed_versions={schema_changes}"),
]
checked_utc = datetime.now(timezone.utc)
records = [
    {
        "run_id": selected_run_id,
        "check_name": name,
        "passed": passed,
        "details": details,
        "checked_utc": checked_utc,
    }
    for name, passed, details in checks
]
reconciliation_schema = spark.read.format("delta").load(reconciliation_path).schema
spark.createDataFrame(records, reconciliation_schema).write.format("delta").mode("append").save(reconciliation_path)
display(spark.createDataFrame(records, reconciliation_schema))

failed_checks = [name for name, passed, _ in checks if not passed]
if failed_checks:
    raise RuntimeError(
        f"STEP 04 FAILED for run {selected_run_id}. Failed checks: {', '.join(failed_checks)}"
    )

summary = {"run_id": selected_run_id, "checks": len(checks), "failed_checks": 0}
print("STEP 04 PASSED")
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary, separators=(",", ":")))
